# Carver Ultimate Daily Multi-Asset Prop Engine v1

### Daily decision framework • Carver time-series + cross-sectional momentum • separated asset buckets • beta rotation • prop drawdown governor

This notebook is the consolidated research architecture.

**Core principle:** the time series is **daily**. Every trading day is one observation and the engine can make a new decision every day. Momentum lookbacks are *not* the timeframe: they are trailing windows measured in daily bars.

The cross-sectional layer follows the supplied Carver material: relative momentum is useful only when the comparison bucket has genuine dispersion/decorrelation. Therefore **stocks, indices and crypto are not demeaned against one another**. Each sleeve has its own peer bucket, and a higher-level allocator combines sleeves.

The notebook is research infrastructure, **not a guarantee of FTMO compliance or future profitability**.


In [ ]:
# =========================
# 0. CONFIGURATION
# =========================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import yfinance as yf
    YF_AVAILABLE = True
except Exception:
    YF_AVAILABLE = False

SEED = 42

# Daily observations
ANN_DAYS_EQUITY = 252
ANN_DAYS_CRYPTO = 365

# Carver forecast conventions
FC_TARGET = 10.0
FC_CAP = 20.0

# Research transaction cost assumption
COST_BPS = 3.25

# Portfolio-level risk target
PORTFOLIO_VOL_TARGET = 0.10

# Prop-oriented hard research ceiling
MAX_RESEARCH_DD = 0.10

# Position constraints
MAX_SINGLE_WEIGHT = 0.20
MIN_POSITION_WEIGHT = 0.00

# Cross-sectional momentum: DAILY bars, multiple horizons
CS_HORIZONS = (21, 63, 126)      # 1M / 3M / 6M in trading days
CS_HORIZON_WEIGHTS = (0.25, 0.35, 0.40)

# Time-series momentum: daily bars, multiple horizons
TS_MOM_HORIZONS = (5, 21, 63, 126, 252)
TS_MOM_WEIGHTS = (0.10, 0.20, 0.25, 0.25, 0.20)

# Trend
TREND_FAST = 50
TREND_SLOW = 200

# Volatility / correlation
VOL_WINDOW = 30
CORR_WINDOW = 90

# Daily signal, next-bar execution. Rebalance every day by default.
REBALANCE_EVERY = 1
EXEC_LAG = 1

# Universe design: keep peer groups separate.
UNIVERSES = {
    "indices": ["SPY", "QQQ", "DIA", "IWM"],
    "stocks": ["AAPL", "MSFT", "AMZN", "GOOGL", "META", "NFLX", "NVDA", "TSLA"],
    "crypto": ["BTC-USD", "ETH-USD", "SOL-USD", "BNB-USD", "XRP-USD", "ADA-USD"],
    "regime": ["XLY", "XLP", "XLU", "SPY", "RSP"],
}

print("Daily engine configured.")
print("CS horizons:", CS_HORIZONS, "daily bars")


## 1. Architecture decision: what is the better momentum design?

The correct distinction is:

- **Timeframe:** daily (`1D`)
- **Signal horizon:** how many daily observations we look back

So `21D` momentum is monthly momentum calculated from daily data. It is **not** a monthly timeframe.

For the final engine we use a **multi-horizon momentum stack** rather than only 1M + 6M:

- 5D = very short-term confirmation
- 21D = monthly
- 63D = quarterly
- 126D = six-month
- 252D = annual context

The cross-sectional Carver layer deliberately uses the more stable **21/63/126D** horizons. This keeps the layer lower-noise and avoids making the relative-ranking component a one-day reversal strategy.

The supplied Carver material's key rule is preserved: **the cross-sectional bucket must contain assets with genuine relative dispersion/decorrelation**. Stocks and crypto are therefore separate CS buckets rather than one giant mixed universe.


In [ ]:
# =========================
# 2. DATA LOADING
# =========================
def synthetic_prices(names, n=3000, seed=SEED, crypto=False):
    rng = np.random.default_rng(seed)
    idx = pd.date_range(end=pd.Timestamp.today().normalize(), periods=n, freq="D")
    k = len(names)

    # Shared factor + rotating leadership + idiosyncratic noise.
    market = np.zeros(n)
    state = 0
    probs = np.array([[0.985, 0.010, 0.005],
                      [0.015, 0.975, 0.010],
                      [0.010, 0.015, 0.975]])
    drift = np.array([0.00045, -0.00055, 0.00008]) if not crypto else np.array([0.00075, -0.00090, 0.00010])
    vol = np.array([0.010, 0.022, 0.014]) if not crypto else np.array([0.025, 0.055, 0.035])

    for t in range(n):
        state = rng.choice(3, p=probs[state])
        market[t] = drift[state] + vol[state] * rng.normal()

    cols = {}
    for j, name in enumerate(names):
        beta = rng.uniform(0.55, 1.35)
        ivol = rng.uniform(0.006, 0.018) if not crypto else rng.uniform(0.015, 0.040)

        # Slow rotating idiosyncratic leader.
        leader = np.sin(np.linspace(0, 18*np.pi, n) + j * 0.9)
        leader_drift = 0.00035 * leader

        r = beta * market + leader_drift + ivol * rng.normal(size=n)
        cols[name.replace("-USD","")] = 100 * np.exp(np.cumsum(r))

    return pd.DataFrame(cols, index=idx)


def load_prices(symbols, start="2016-01-01", source="auto", seed=SEED):
    clean = [s.replace("-USD", "") for s in symbols]

    if source in ("auto", "yfinance") and YF_AVAILABLE:
        try:
            raw = yf.download(symbols, start=start, auto_adjust=True,
                               progress=False, group_by="column", threads=True)
            if isinstance(raw.columns, pd.MultiIndex):
                if "Close" in raw.columns.get_level_values(0):
                    df = raw["Close"].copy()
                else:
                    df = raw.xs("Close", axis=1, level=-1)
            else:
                df = raw[["Close"]].copy()
                df.columns = [symbols[0]]

            df.columns = [str(c).replace("-USD", "") for c in df.columns]
            df = df.sort_index().ffill().dropna(how="all")
            df = df.dropna(axis=1, thresh=max(100, int(len(df)*0.60)))
            if len(df) >= 500 and df.shape[1] >= 3:
                return df
            raise RuntimeError("insufficient common data")
        except Exception as e:
            if source == "yfinance":
                raise
            print("yfinance fallback:", repr(e))

    return synthetic_prices(clean, seed=seed, crypto=any("-USD" in s for s in symbols))


panels = {}
for bucket in ("indices", "stocks", "crypto"):
    panels[bucket] = load_prices(UNIVERSES[bucket], seed=SEED + len(bucket))

print({k: v.shape for k,v in panels.items()})


## 3. Risk engine

We use a Carver-style volatility stack:

1. daily returns
2. short EWMA volatility
3. long volatility anchor
4. blended annualized volatility
5. volatility-normalized price path

This makes the cross-sectional comparison meaningful across assets with very different raw prices and volatility.

**Annualization matters:** equity/index sleeves use 252 observations/year; crypto uses 365.


In [ ]:
def ewm_std(x, span):
    a = 2.0 / (span + 1.0)
    m = x.ewm(alpha=a, adjust=False).mean()
    m2 = (x*x).ewm(alpha=a, adjust=False).mean()
    var = (m2 - m*m).clip(lower=0)
    bc = (2.0 - 2.0*a) / (2.0 - a)
    return np.sqrt(var / bc)


def vol_stack(close, ann_days):
    ret = close.pct_change().clip(-0.9, 9.0)
    short = ewm_std(ret, VOL_WINDOW) * np.sqrt(ann_days)
    anchor = short.rolling(5*ann_days, min_periods=ann_days).mean()
    vol = pd.Series(
        np.where(anchor.isna(), short, 0.70*short + 0.30*anchor),
        index=close.index
    ).clip(lower=1e-6)
    sigma_p = close * vol / np.sqrt(ann_days)
    return pd.DataFrame({"ret":ret, "vol":vol, "sigma_p":sigma_p})


def max_dd(eq):
    return float((eq / eq.cummax() - 1).min())


def sharpe(ret, ann_days):
    sd = ret.std(ddof=1)
    return np.nan if sd == 0 else float(ret.mean()/sd*np.sqrt(ann_days))


def cagr(eq):
    if len(eq) < 2 or eq.iloc[-1] <= 0:
        return np.nan
    years = (eq.index[-1] - eq.index[0]).days / 365.25
    return float(eq.iloc[-1]**(1/years)-1) if years > 0 else np.nan


def performance(name, result, ann_days):
    print(f"{name:28s}  Sharpe={sharpe(result['net'],ann_days):5.2f}  "
          f"CAGR={cagr(result['equity'])*100:6.1f}%  "
          f"MaxDD={max_dd(result['equity'])*100:6.1f}%  "
          f"Turn={result['turnover'].sum()/max(1,len(result)/ann_days):5.1f}x")


## 4. Daily time-series forecast stack

The engine does **not** rely on one momentum period.

The time-series component combines:

- EWMAC
- breakout
- acceleration
- skew
- multi-horizon momentum
- trend confirmation

All signals are calculated on the daily series. Forecasts are clipped to ±20 and later converted to volatility-adjusted positions.


In [ ]:
EWMAC_PAIRS = [(8,32,5.95),(16,64,4.10),(32,128,2.79),(64,256,1.91)]

def clip_fc(x):
    return x.clip(-FC_CAP, FC_CAP)

def ewmac_forecast(close, sigma_p):
    out=[]
    for f,s,sc in EWMAC_PAIRS:
        raw=(close.ewm(span=f,adjust=False).mean()-close.ewm(span=s,adjust=False).mean())/sigma_p
        out.append((raw*sc).clip(-FC_CAP,FC_CAP))
    return (sum(out)/len(out)*1.13).clip(-FC_CAP,FC_CAP)

def breakout_forecast(close):
    parts=[]
    for n,sc,sm in [(40,.70,10),(80,.73,20),(160,.74,40),(320,.74,80)]:
        lo=close.rolling(n).min()
        hi=close.rolling(n).max()
        rng=(hi-lo).replace(0,np.nan)
        pos=((close-lo)/rng).fillna(.5)
        parts.append(((pos-.5)*40).ewm(span=sm,adjust=False).mean()*sc)
    return (sum(parts)/len(parts)*1.17).clip(-FC_CAP,FC_CAP)

def acceleration_forecast(close,sigma_p):
    bases=[]
    for f,s,sc in EWMAC_PAIRS:
        bases.append(((close.ewm(span=f,adjust=False).mean()-
                       close.ewm(span=s,adjust=False).mean())/sigma_p*sc).clip(-FC_CAP,FC_CAP))
    aps=[8,16,32,64]
    scales=[1.87,1.90,1.98,2.05]
    parts=[(bases[i]-bases[i].shift(aps[i]))*scales[i] for i in range(4)]
    return (sum(parts)/len(parts)*1.55).clip(-FC_CAP,FC_CAP)

def skew_forecast(ret):
    parts=[]
    for w,sc,sm in [(60,33.3,15),(120,37.2,30),(240,39.2,60)]:
        g=ret.rolling(w,min_periods=w//2).skew()
        parts.append((-g).ewm(span=sm,adjust=False).mean()*sc)
    return (sum(parts)/len(parts)*1.18).clip(-FC_CAP,FC_CAP)

def multihorizon_momentum(close):
    parts=[]
    for h,w in zip(TS_MOM_HORIZONS,TS_MOM_WEIGHTS):
        r=(close/close.shift(h)-1)
        # volatility-normalize the horizon return
        parts.append(r * w)
    raw=sum(parts)
    return (raw / raw.rolling(252,min_periods=63).std()).clip(-3,3) * (FC_TARGET/2)

def trend_score(close):
    fast=close.ewm(span=TREND_FAST,adjust=False).mean()
    slow=close.ewm(span=TREND_SLOW,adjust=False).mean()
    return ((fast/slow)-1).clip(-0.10,0.10) * 100

def vol_attenuation(vol):
    pct=vol.rolling(1260,min_periods=252).rank(pct=True)
    return (1.5-pct.ewm(span=10,adjust=False).mean()).clip(.5,1.5).fillna(1.0)

def build_ts_forecasts(close, ann_days):
    vs=vol_stack(close,ann_days)
    ew=ewmac_forecast(close,vs.sigma_p)
    br=breakout_forecast(close)
    ac=acceleration_forecast(close,vs.sigma_p)
    sk=skew_forecast(vs.ret)
    mom=multihorizon_momentum(close)
    tr=trend_score(close)

    # 5-family forecast; trend is a confirmation term rather than a standalone high-weight strategy.
    combined=(0.25*ew + 0.20*br + 0.15*ac + 0.15*sk + 0.15*mom + 0.10*tr)
    combined=clip_fc(combined*vol_attenuation(vs.vol))

    # Long-only: negative forecast becomes zero.
    long_fc=combined.clip(lower=0)

    return {
        "combined": combined,
        "long_fc": long_fc,
        "vol": vs.vol,
        "ret": vs.ret,
        "sigma_p": vs.sigma_p,
        "trend": tr,
        "momentum": mom,
        "EWMAC": ew,
        "Breakout": br,
        "Accel": ac,
        "Skew": sk
    }


## 5. Cross-sectional momentum — the Carver layer

This is the part that should **not** be confused with ordinary absolute momentum.

For each peer bucket:

`normalized price → asset-class average → relative price → trailing outperformance → forecast`

The relative forecast answers:

> Is this asset outperforming the *other assets in its own bucket*?

The bucket is therefore crucial. We explicitly avoid one cross-sectional universe containing stocks + crypto because the supplied Carver material warns against mixing materially different / overlapping groups and emphasizes genuine decorrelation.


In [ ]:
def normalized_price(close, ann_days):
    vs=vol_stack(close,ann_days)
    rn=(vs.ret/(vs.vol/np.sqrt(ann_days))).replace([np.inf,-np.inf],np.nan).fillna(0)
    return 100*rn.cumsum()

def cross_sectional_forecasts(panel, ann_days, horizons=CS_HORIZONS):
    pn=pd.DataFrame({c:normalized_price(panel[c],ann_days) for c in panel.columns})
    asset_class=pn.mean(axis=1)
    result={}

    for target in panel.columns:
        rel=pn[target]-asset_class
        subs=[]
        for h in horizons:
            raw=(rel-rel.shift(h))/h
            sm=raw.ewm(span=max(2,h//4),adjust=False).mean()
            # Fixed-style calibration using a trailing broad history rather than a single recent point.
            scale=10.0/max(sm.abs().rolling(756,min_periods=126).mean().median(),1e-6)
            subs.append(sm*scale)

        cs=sum(w*s for w,s in zip(CS_HORIZON_WEIGHTS,subs))
        result[target]=cs.clip(-FC_CAP,FC_CAP)

    return pd.DataFrame(result), pn, asset_class


def forecast_fdm(forecasts):
    F=forecasts.dropna(how="all")
    C=F.corr().fillna(0)
    np.fill_diagonal(C.values,1.0)
    n=C.shape[0]
    w=np.ones(n)/n
    var=float(w @ C.values @ w)
    return float(min(2.5,1/np.sqrt(max(var,1e-8))))


def bucket_scores(panel, ann_days):
    ts={a:build_ts_forecasts(panel[a],ann_days) for a in panel.columns}
    cs,_,_=cross_sectional_forecasts(panel,ann_days)

    rows=[]
    for a in panel.columns:
        x=ts[a]
        # Combine absolute trend/momentum with relative strength.
        score=(0.55*x["long_fc"] + 0.45*cs[a]).clip(lower=0)
        eligible=(x["trend"]>0) & (x["momentum"]>0) & (cs[a]>0)
        score=score.where(eligible,0.0)
        rows.append(pd.DataFrame({
            "score":score,
            "vol":x["vol"],
            "trend":x["trend"],
            "momentum":x["momentum"],
            "cs":cs[a]
        }).assign(symbol=a))

    return pd.concat(rows,axis=0)


# Example on the stock bucket
stock_scores=bucket_scores(panels["stocks"],ANN_DAYS_EQUITY)
print(stock_scores.tail())


## 6. Portfolio construction inside each bucket

We select leaders using:

- positive long-only time-series signal
- positive multi-horizon momentum
- positive cross-sectional relative strength
- inverse-volatility sizing
- correlation penalty
- 20% single-name cap

The result is a **leader portfolio**, not a collection of identical bets.


In [ ]:
def make_bucket_weights(panel, ann_days, n_assets=4, max_weight=MAX_SINGLE_WEIGHT):
    score_df=bucket_scores(panel,ann_days)
    idx=panel.index
    symbols=list(panel.columns)
    out=pd.DataFrame(0.0,index=idx,columns=symbols)

    # Cross-sectional rank is recalculated every day.
    for t in idx:
        d=score_df.xs(t,level=0) if t in score_df.index.get_level_values(0) else None
        if d is None or len(d)==0:
            continue

        d=d[d["score"]>0].copy()
        if d.empty:
            continue

        # Correlation penalty from trailing returns.
        loc=idx.get_loc(t)
        start=max(0,loc-CORR_WINDOW+1)
        hist=panel.iloc[start:loc+1].pct_change().dropna()
        corr=hist.corr().abs()

        candidates=d.sort_values("score",ascending=False).head(n_assets)
        raw={}
        for sym,row in candidates.iterrows():
            base=float(row["score"])/max(float(row["vol"]),1e-6)
            others=[x for x in candidates.index if x!=sym and x in corr.columns]
            avgcorr=float(corr.loc[sym,others].mean()) if others else 0.0
            penalty=np.clip(1-avgcorr,0.25,1.0)
            raw[sym]=base*penalty

        raw=pd.Series(raw,dtype=float)
        if raw.sum()>0:
            w=raw/raw.sum()
            # Iterative cap-and-redistribute.
            for _ in range(10):
                over=w>w.max()*0+max_weight
                if not over.any():
                    break
                excess=(w[over]-max_weight).sum()
                w[over]=max_weight
                under=w<max_weight
                if under.any():
                    w[under]+=excess*w[under]/w[under].sum()
                else:
                    break
            w=w/w.sum()
            out.loc[t,w.index]=w.values

    return out.fillna(0.0)


bucket_weights={}
for bucket,ann in [("indices",ANN_DAYS_EQUITY),("stocks",ANN_DAYS_EQUITY),("crypto",ANN_DAYS_CRYPTO)]:
    bucket_weights[bucket]=make_bucket_weights(panels[bucket],ann)
    print(bucket, bucket_weights[bucket].tail(1).round(3).to_dict("records")[0])


## 7. Beta / regime rotation

The regime overlay is intentionally simple.

Risk-on evidence:
- XLY / XLP
- RSP / SPY

Risk-off evidence:
- XLU / SPY (inverted)

The regime overlay **scales long exposure**. It does not create short positions.

This is a portfolio-level overlay, so it can be applied after the individual sleeves have already done their own cross-sectional selection.


In [ ]:
def zscore(s,window=126):
    m=s.rolling(window,min_periods=63).mean()
    sd=s.rolling(window,min_periods=63).std()
    return ((s-m)/sd.replace(0,np.nan)).clip(-3,3)

def beta_regime(regime_panel):
    p=regime_panel.copy().dropna(how="all").ffill().dropna()
    xly_xlp=p["XLY"]/p["XLP"]
    xlu_spy=p["XLU"]/p["SPY"]
    rsp_spy=p["RSP"]/p["SPY"]

    score=(zscore(xly_xlp)+zscore(rsp_spy)-zscore(xlu_spy))/3
    state=pd.Series("NEUTRAL",index=score.index)
    state[score>=0.50]="RISK_ON"
    state[score<=-0.50]="RISK_OFF"

    scale=state.map({"RISK_ON":1.0,"NEUTRAL":0.65,"RISK_OFF":0.35})
    return pd.DataFrame({"score":score,"state":state,"scale":scale})

# Load regime panel separately
regime=load_prices(UNIVERSES["regime"],seed=99)
regime.columns=[c.replace("-USD","") for c in regime.columns]
reg=beta_regime(regime)
print(reg.tail())


## 8. Portfolio-level allocator

We **do not** make one cross-sectional ranking containing every asset.

Instead:

1. indices sleeve ranks indices
2. stocks sleeve ranks stocks
3. crypto sleeve ranks crypto
4. the regime layer scales total risk
5. a portfolio-level correlation/risk check prevents one sleeve from dominating

This respects the Carver bucket principle while still allowing one overall portfolio to contain multiple asset classes.


In [ ]:
def align_frames(frames):
    idx=frames[0].index
    for f in frames[1:]:
        idx=idx.intersection(f.index)
    return [f.reindex(idx).fillna(0) for f in frames]

def combine_sleeves(bucket_weights, regime, sleeve_caps=None):
    if sleeve_caps is None:
        sleeve_caps={"indices":0.50,"stocks":0.50,"crypto":0.25}

    frames=[]
    for b,w in bucket_weights.items():
        x=w.copy()*sleeve_caps.get(b,1.0)
        frames.append(x)

    idx=frames[0].index
    for f in frames[1:]:
        idx=idx.union(f.index)
    total=pd.DataFrame(0.0,index=idx)
    for f in frames:
        total=total.add(f.reindex(idx).fillna(0),fill_value=0)

    rs=regime["scale"].reindex(idx).ffill().fillna(0.65)
    total=total.mul(rs,axis=0)

    # Portfolio cash is whatever remains.
    gross=total.sum(axis=1)
    scale=(1.0/gross).where(gross>1,1.0)
    total=total.mul(scale,axis=0)
    cash=1-total.sum(axis=1)

    return total.clip(lower=0),cash

portfolio_weights,cash_weight=combine_sleeves(bucket_weights,reg)
print("Latest portfolio weights:")
print(portfolio_weights.tail(1).T[portfolio_weights.tail(1).index[0]].sort_values(ascending=False).head(15))
print("Cash:", float(cash_weight.iloc[-1]))


## 9. Progressive drawdown governor

The research portfolio gets a separate survival layer.

- DD < 2.5% → 100% of target exposure
- DD 2.5–5% → 75%
- DD 5–7.5% → 50%
- DD 7.5–10% → 25%
- DD ≥ 10% → 0% risk exposure

This is a **risk governor**, not a promise that a live account can never breach a firm's rules. Slippage, gaps, spreads and execution can exceed modelled losses.


In [ ]:
def portfolio_return(close_panels, weights):
    # Union all prices; forward-fill within each series only.
    prices=pd.concat(close_panels,axis=1).sort_index()
    prices=prices.loc[:,~prices.columns.duplicated()]
    rets=prices.pct_change().fillna(0)
    common=weights.index.intersection(rets.index)
    return (weights.reindex(common).fillna(0)*rets.reindex(common)[weights.columns]).sum(axis=1)

def dd_governor(strategy_ret):
    eq=(1+strategy_ret.fillna(0)).cumprod()
    dd=eq/eq.cummax()-1
    scale=pd.Series(1.0,index=dd.index)
    scale[dd<=-0.025]=0.75
    scale[dd<=-0.050]=0.50
    scale[dd<=-0.075]=0.25
    scale[dd<=-0.100]=0.00
    return scale,dd,eq

# Raw portfolio return using next-day held weights.
all_prices=pd.concat([panels["indices"],panels["stocks"],panels["crypto"]],axis=1)
all_prices=all_prices.loc[:,~all_prices.columns.duplicated()]
aligned_w=portfolio_weights.reindex(all_prices.index).fillna(0)
held_w=aligned_w.shift(EXEC_LAG).fillna(0)
rets=all_prices.pct_change().fillna(0)

raw_ret=(held_w[all_prices.columns]*rets[all_prices.columns]).sum(axis=1)
turnover=held_w.diff().abs().sum(axis=1).fillna(held_w.abs().sum(axis=1))
raw_net=raw_ret-turnover*COST_BPS/1e4

dd_scale,dd,raw_eq=dd_governor(raw_net)
final_ret=raw_ret*dd_scale-turnover*COST_BPS/1e4
final_eq=(1+final_ret).cumprod()

print("Raw MaxDD:", round(max_dd(raw_eq)*100,2), "%")
print("Governed MaxDD:", round(max_dd(final_eq)*100,2), "%")


## 10. Backtest diagnostics

The objective is **not** to maximize CAGR.

For prop deployment the hierarchy is:

1. maximum drawdown / survival
2. recovery time
3. consistency
4. low concentration
5. controlled turnover
6. Sharpe / Calmar
7. CAGR

The notebook therefore reports both return and survival statistics.


In [ ]:
def recovery_days(eq):
    dd=eq/eq.cummax()-1
    underwater=dd<0
    max_days=0
    cur=0
    for x in underwater:
        cur=cur+1 if x else 0
        max_days=max(max_days,cur)
    return max_days

def tail_loss(ret,q=0.05):
    return float(ret.quantile(q))

def diagnostics(name, ret, eq, ann_days):
    dd=eq/eq.cummax()-1
    years=max((eq.index[-1]-eq.index[0]).days/365.25,1e-9)
    return {
        "Strategy":name,
        "CAGR":cagr(eq),
        "Sharpe":sharpe(ret,ann_days),
        "MaxDD":float(dd.min()),
        "Calmar":float(cagr(eq)/abs(dd.min())) if dd.min()<0 else np.nan,
        "WorstDay":float(ret.min()),
        "P5":tail_loss(ret),
        "RecoveryDays":recovery_days(eq),
        "AvgExposure":float((1-cash_weight.reindex(ret.index).fillna(1)).mean())
    }

metrics=pd.DataFrame([
    diagnostics("RAW",raw_net,raw_eq,ANN_DAYS_EQUITY),
    diagnostics("DD GOVERNOR",final_ret,final_eq,ANN_DAYS_EQUITY)
])
print(metrics.to_string(index=False))


## 11. A/B tests the final engine should pass before deployment

The notebook intentionally keeps the main parameters centralized so you can test:

- Carver TS only vs TS + CS
- 21/63/126 CS vs 21/126 CS
- daily vs weekly rebalance
- no regime vs beta regime
- no drawdown governor vs governor
- equal weight vs inverse volatility
- indices only / stocks only / crypto only / combined sleeves
- different maximum single-name weights
- different portfolio volatility targets

Do **not** select parameters from the best in-sample result. Use a train / validation / out-of-sample or walk-forward protocol.


In [ ]:
# =========================
# 12. QUICK ROBUSTNESS GRID
# =========================
def apply_rebalance_frequency(weights, every_n=1):
    if every_n<=1:
        return weights.copy()
    out=weights.copy()
    mask=np.arange(len(out))%every_n==0
    out.loc[~mask]=np.nan
    return out.ffill().fillna(0)

def simple_backtest(prices, weights, ann_days=252, cost_bps=COST_BPS):
    held=weights.shift(1).fillna(0)
    r=prices.pct_change().fillna(0)
    turnover=held.diff().abs().sum(axis=1).fillna(held.abs().sum(axis=1))
    net=(held[prices.columns]*r[prices.columns]).sum(axis=1)-turnover*cost_bps/1e4
    eq=(1+net).cumprod()
    return net,eq,turnover

grid=[]
for freq in [1,5,10]:
    w=apply_rebalance_frequency(portfolio_weights,freq)
    net,eq,to=simple_backtest(all_prices.reindex(w.index).ffill(),w)
    grid.append({
        "RebalanceDays":freq,
        "CAGR":cagr(eq),
        "Sharpe":sharpe(net,ANN_DAYS_EQUITY),
        "MaxDD":max_dd(eq),
        "Turnover":to.sum()/max(1,len(to)/ANN_DAYS_EQUITY)
    })

print(pd.DataFrame(grid).round(4).to_string(index=False))


## 13. Walk-forward / anti-overfitting protocol

For the final research version:

**Never** calibrate the cross-sectional scalar, thresholds or weights on the entire history and then report that history as proof.

Recommended process:

1. Train: earliest 60%
2. Validation: next 20%
3. Test: final 20%
4. Then perform rolling walk-forward tests
5. Freeze parameters before each out-of-sample segment
6. Include transaction costs and next-bar execution
7. Include delisted constituents for a proper historical stock universe where possible

The supplied Carver material specifically notes survivorship bias as a danger for stock cross-sectional tests and says hard-coded calibration constants should ultimately be fixed from a broad out-of-sample panel.


## 14. Final architecture

```text
                         DAILY MARKET DATA
                                │
             ┌──────────────────┴──────────────────┐
             │                                     │
      TIME-SERIES ENGINE                    PEER BUCKET ENGINE
      EWMAC / Breakout                      INDICES
      Acceleration / Skew                   STOCKS
      Multi-horizon momentum                CRYPTO
      Trend filter                          (separate CS buckets)
             │                                     │
             └──────────────────┬──────────────────┘
                                │
                         FORECAST BLEND
                                │
                         CORRELATION/FDM
                                │
                    INVERSE-VOL + CORR PENALTY
                                │
                     SINGLE-NAME CAP = 20%
                                │
                       BETA REGIME ROTATION
                                │
                       PORTFOLIO VOL TARGET
                                │
                    PROGRESSIVE DD GOVERNOR
                                │
                         NEXT-DAY EXECUTION
                                │
                         CASH WHEN NEEDED
```

### Bottom line

**This is the version I would use as the research master.**

It resolves the earlier ambiguity:

- **1D is the data/decision timeframe.**
- Momentum is **multi-horizon daily momentum**, not a monthly timeframe.
- Cross-sectional momentum is retained because it can add diversification, but only inside **intentional peer buckets**.
- Stocks + crypto are **not** one cross-sectional bucket.
- Long-only is the default for the spot-like research universe.
- Beta rotation is an overlay, not a replacement for the core signal.
- Drawdown governance is portfolio-level and progressive.
- Daily signal generation and next-bar execution are explicit.
- The system is designed around **survival and robustness**, not maximum backtest CAGR.

**Important:** this notebook is a research engine. Before any prop deployment, verify the current firm's exact daily-loss, max-loss, overnight/weekend, instrument and exposure rules separately.
